# OASIS-2 Minimal Colab Training

This notebook is a clean Colab-first path for OASIS-2 training.
It clones the repo, installs dependencies, runs the OASIS-2 training pipeline with live logs, and prints the saved summary.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_baseline_v2'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = '8b847ca'

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print('repo_commit =', repo_commit)
print('required_commit =', REQUIRED_COMMIT)



In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '20',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '1',
    '--num-workers', '0',
    '--image-size', '64', '64', '64',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
]

print('RUNNING:', ' '.join(cmd), flush=True)
process = subprocess.Popen(
    cmd,
    cwd=BACKEND_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'OASIS-2 training runner failed with exit code {return_code}')

print('OASIS-2 training runner completed successfully.', flush=True)


In [ ]:
from pathlib import Path
import json
import pandas as pd

blocked_summary_path = RUNTIME_ROOT / 'outputs' / 'reports' / 'onboarding' / 'oasis2_colab_bundle_summary.json'
trained_summary_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'reports' / 'colab_run_summary.json'

print('blocked_summary_path =', blocked_summary_path)
print('trained_summary_path =', trained_summary_path)

summary_candidates = [trained_summary_path, blocked_summary_path]
existing_summaries = [path for path in summary_candidates if path.exists()]
if not existing_summaries:
    raise FileNotFoundError('Expected an OASIS-2 summary JSON, but none was found in the runtime outputs.')

summary_path = max(existing_summaries, key=lambda path: path.stat().st_mtime)
print('using_summary_path =', summary_path)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
print('training_ready =', summary.get('training_ready'))
print('training_started =', summary.get('training_started'))
print('blocked_reason =', summary.get('blocked_reason'))
print('run_root =', summary.get('run_root'))
print('best_checkpoint =', summary.get('best_checkpoint'))

metrics_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'metrics' / 'epoch_metrics.csv'
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print(metrics.tail())
